# 02 · Labels: the 58 gold studies and soft labels derived from the reports

**What this notebook does:** it prepares the 12 finding labels used for training and measures how reliable
they are. Its output is `data/train_labels*.csv`, read by the training notebooks.

**Why a separate step?** The real difficulty of this competition is the labels, not the images:

| | studies | have the 12 labels | have a report |
|---|---:|:---:|:---:|
| `train.csv` | 4,407 | **only 58** | all |
| `test.csv` | hidden | — | **no** |

* For the 4,349 unlabelled studies the labels must be extracted from the reports (multilingual free text).
* The test set has no reports, so text cannot be used at prediction time; it only produces **training
  targets**. "Text + image fusion" is therefore not possible here.
* The 58 gold studies are the only ground truth for choosing label sources and validating models, but 58 is
  very few: small differences are statistical noise (measured below).

**Module:** `src/labels.py`

In [ ]:
# Add the project root (the folder that contains src/) to the Python path; works locally and on Kaggle.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk with pruning: never enters the competition DICOM folders (rglob took minutes)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ not found")

PROJECT_ROOT = add_project_root()
print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_DIR, TARGET_COLS
from src import labels as L

pd.set_option("display.precision", 3)
gold = L.load_gold()
print(f"Gold studies: {len(gold)}")

## 1. Label sources

Public label sets (the competition rules allow public external data):

| name | Kaggle dataset | file | how it was made |
|---|---|---|---|
| `steven_v4` | `stevenleehans/rsna-knee-llm-report-labels` | `llm_labels_v4_blend.csv` | LLM reads + blend |
| `pilkwang_v2` | `pilkwang/rsna-knee-llm-labels` | `report_labels_v2.csv` | lexicon rules + an LLM second reader |
| `opus45` | github.com/NTejas-1/RSNA-Knee-Abnormality-Detection (MIT) | `labels_v1_opus45.csv` | Claude Opus 4.5 (added 10-14) |
| `jev_hybrid` | `riadmohamed42/jev-knee-labels` | `HYBRID_labels_scores.csv` | steven + pilkwang + a text model (added 10-14) |

First, the ROC AUC of each source on the 58 gold studies. It answers "how well does a report reader rank the
radiologist's labels?" 

In [ ]:
ALL_SOURCES = {**L.SOURCES_V1, **L.SOURCES_V2}
audit = L.audit_sources(ALL_SOURCES)
audit.style.background_gradient(axis=None, cmap="RdYlGn", vmin=0.65, vmax=1.0).format("{:.3f}")

**Reading:** findings that reports state clearly (ACL, MCL, Baker's) reach ~0.95+; **Synovitis, Fracture,
Lateral OA and Contusion** are weak. Reports describe them indirectly ("bone marrow oedema", "synovial
thickening", ...) and the language coverage is incomplete. These rows largely set the ceiling of any model
trained on report labels.

## 2. Positive rates: do the sources agree with the gold studies?

In [ ]:
src = {k: L.load_source(f) for k, f in ALL_SOURCES.items()}
prev = pd.DataFrame({"gold (58)": gold.mean()})
for k, df in src.items():
    prev[k + " (>0.5)"] = (df > 0.5).mean()
ax = prev.plot.bar(figsize=(13, 4), rot=30, title="Positive rate per finding")
ax.set_ylabel("oran"); plt.tight_layout(); plt.show()
prev.T

**Reading:** report-based labels mark Fracture, Synovitis and Lateral Meniscus clearly **less often** than the
gold studies. AUC only looks at the ranking, so this calibration gap does not break the metric directly, but
* rate-based settings such as `pos_weight` become biased,
* binarising labels at 0.5 for validation shrinks the positive class of these findings.

So training uses plain BCE on the soft labels, without a fixed `pos_weight`.

## 3. How similar are the two sources?

In [ ]:
ng = src["steven_v4"].index.difference(gold.index)
corr = pd.Series({c: np.corrcoef(src["steven_v4"].loc[ng, c],
                                  src["pilkwang_v2"].reindex(ng)[c].fillna(0.5))[0, 1]
                  for c in TARGET_COLS}, name="korelasyon")
corr.to_frame().T

The correlation is 0.81–0.98: the sources mostly agree but diverge on Contusion and Fracture. Averaging two
independent readers partly cancels their individual errors.

## 4. Does averaging really help? (the noise of 58 studies)

The variability of a 58-study AUC is measured with a **bootstrap**: resample the studies with replacement and
recompute the difference 300 times. If the interval contains zero, the difference is not real.

In [ ]:
from sklearn.metrics import roc_auc_score

blend = (src["steven_v4"] + src["pilkwang_v2"].reindex(src["steven_v4"].index).fillna(src["steven_v4"])) / 2
def macro(idx, pred):
    return np.mean([roc_auc_score(gold.loc[idx, c], pred.loc[idx, c]) for c in TARGET_COLS])

rng = np.random.default_rng(0)
deltas = []
for _ in range(300):
    idx = rng.choice(gold.index, len(gold))
    try:
        deltas.append(macro(idx, blend) - macro(idx, src["steven_v4"]))
    except ValueError:  # skip resamples where a finding has a single class
        pass
lo, hi = np.percentile(deltas, [5, 95])
print(f"blend - steven_v4 : mean {np.mean(deltas):+.4f}, 90% interval [{lo:+.4f}, {hi:+.4f}]")
print(f"blend macro AUC (58): {macro(gold.index, blend):.4f}")

**Result:** the difference is inside the noise band. 58 studies cannot tell the two sources apart, so "pick the
source with the best gold score" is effectively a random choice and leaks the validation set.

**Decision:** an equal-weight mean of the sources. The reason is structural, not statistical: averaging
independent readers lowers the variance, and no source is tuned on the gold studies.

## 5. Build the training label table

* Unlabelled studies: the mean of the sources (soft labels in 0–1).
* The 58 gold studies: the radiologist labels (0/1), `is_gold=True`; they get 2× weight in training and are
  reported separately in validation.

In [ ]:
train_labels = L.build_training_labels(L.SOURCES_V1)
out_path = DATA_DIR / "train_labels.csv"
train_labels.to_csv(out_path, index=False)
print(f"Kaydedildi: {out_path}  ->  {train_labels.shape}")
print(f"Gold studies: {train_labels.is_gold.sum()}  |  empty cells: {train_labels[TARGET_COLS].isna().sum().sum()}")
train_labels.head()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.5))
train_labels.loc[~train_labels.is_gold, TARGET_COLS].plot.box(ax=ax, rot=30)
ax.set_title("Soft-label distribution (4,349 unlabelled studies)")
plt.tight_layout(); plt.show()

## 6. Labels v2 (2026-10-14): a third and a fourth reader

Two strong public sources were found on the forum: Claude Opus 4.5 labels and JEV HYBRID.
JEV already contains steven and pilkwang, so pilkwang is not added again (it would be counted twice).
Macro AUC of the possible blends on the 58 gold studies:

In [ ]:
import itertools
names = ["steven_v4", "pilkwang_v2", "opus45", "jev_hybrid"]
mean_all = pd.concat([src[n] for n in names]).groupby(level=0).mean()
rows = []
for r in range(1, 5):
    for combo in itertools.combinations(names, r):
        b = sum(src[n].reindex(mean_all.index).fillna(mean_all) for n in combo) / len(combo)
        rows.append({"blend": " + ".join(combo), "macro AUC": L.macro_auc(gold, b)["macro"]})
pd.DataFrame(rows).sort_values("macro AUC", ascending=False).head(8).round(4)

**Decision:** labels v2 = steven_v4 + opus45 + jev_hybrid (0.9014). The differences are close to the noise of
58 studies, but a mean of three independent readers is structurally more robust. Our own Qwen-7B labelling
(scripts/llm_label.py) reached only 0.797 and was not added. Note: the gold labels sometimes contradict the
report (the radiologist looked at the images), so report-based labels top out around ~0.90.

In [ ]:
train_labels_v2 = L.build_training_labels(L.SOURCES_V2)
train_labels_v2.to_csv(DATA_DIR / "train_labels_v2.csv", index=False)
print("Kaydedildi:", DATA_DIR / "train_labels_v2.csv", train_labels_v2.shape)

## Summary

| decision | reason |
|---|---|
| v1: 50/50 mean of two public sources (0.8927) | 58 studies cannot separate the sources; averaging reduces noise |
| **v2: steven + Opus 4.5 + JEV (0.9014)** | three independent readers; used by notebooks 05, 07 and 12 |
| v3 / v4: + out-of-fold predictions of our image models (`src/labels.py`) | gold 0.922, but only +0.001 on the leaderboard |
| Gold studies keep their 0/1 labels, 2× weight | the only real signal |
| No `pos_weight` | the positive rates of the sources are biased |
| `gold_auc` reported separately in validation | agreeing with LLM labels ≠ agreeing with radiologists |

**Next:** `03_dicom_preprocessing.ipynb`, reading the images correctly.